In [1]:
# !pip install convokit zstandard
# !pip uninstall -y numpy scikit-learn transformers
# !pip install numpy==1.24.4 scikit-learn==1.3.2 transformers==4.36.2
!nvidia-smi

Sun May 11 10:03:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4000               Off |   00000000:00:05.0 Off |                  Off |
| 41%   53C    P8             16W /  140W |    2993MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from convokit import Corpus, download
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
from torch.optim import AdamW
import random
from datasets import load_dataset
from tqdm import tqdm
import random
from collections import defaultdict


In [3]:
tokenizer = GPT2TokenizerFast.from_pretrained("fine-tuned-gpt2-speakers-8")
model = GPT2LMHeadModel.from_pretrained("fine-tuned-gpt2-speakers-8")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id


# special_tokens = ["<SPK1>", "<SPK2>"]
# tokenizer.add_tokens(special_tokens)
# model.resize_token_embeddings(len(tokenizer))

# Optional: average embedding init
# with torch.no_grad():
#     embedding = model.get_input_embeddings()
#     avg = embedding.weight[:-len(special_tokens)].mean(dim=0)
#     for tok in special_tokens:
#         idx = tokenizer.convert_tokens_to_ids(tok)
#         embedding.weight[idx] = avg


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [4]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Freeze everything except embeddings
for param in model.parameters():
    param.requires_grad = False
model.transformer.wte.weight.requires_grad = True

optimizer = AdamW(model.parameters(), lr=1e-4)


In [5]:
dialogs = []


# corpus = Corpus(filename=download("switchboard-processed-corpus"))

# for convo in tqdm(corpus.iter_conversations(), desc="Parsing conversations"):
#     utterances = list(convo.iter_utterances())
#     if len(utterances) < 2:
#         continue

#     speakers = list({utt.speaker.id for utt in utterances})
#     if len(speakers) != 2:
#         continue
    
#     if random.random() > 0.5:
#         speaker_map = {speakers[0]: "<SPK1>", speakers[1]: "<SPK2>"}
#     else:
#         speaker_map = {speakers[0]: "<SPK2>", speakers[1]: "<SPK1>"}
    
#     dialog = []

#     for utt in utterances:
#         text = utt.text.strip().replace("\n", " ")
#         if text:
#             dialog.append(f"{speaker_map[utt.speaker.id]} {text}")

#     if dialog:
#         dialogs.append(" ".join(dialog))


# ################################

# dataset = load_dataset("daily_dialog", trust_remote_code=True)

# for sample in tqdm(dataset["train"], desc="Parsing DailyDialog"):
#     utterances = sample["dialog"]
#     if len(utterances) < 2:
#         continue

#     dialog = []
#     flip = random.random() > 0.5  # randomly assign SPK1/SPK2
#     for i, text in enumerate(utterances):
#         spk = "<SPK1>" if (i % 2 == 0) ^ flip else "<SPK2>"
#         dialog.append(f"{spk} {text.strip()}")

#     dialogs.append(" ".join(dialog))

# ###########################################

dataset = load_dataset("agentlans/Conversational-Reasoning-Topical-Chat")

for sample in tqdm(dataset["train"], desc="Parsing TopicalChat"):
    dialog = []
    turns = sample['conversations'][1:]

    # Skip single-utterance examples
    if len(turns) < 2:
        continue

    # Identify both speakers and assign SPK1/SPK2 randomly
    all_speakers = list(set(turn["from"] for turn in turns))
    if len(all_speakers) != 2:
        continue

    if random.random() > 0.5:
        speaker_map = {all_speakers[0]: "<SPK1>", all_speakers[1]: "<SPK2>"}
    else:
        speaker_map = {all_speakers[0]: "<SPK2>", all_speakers[1]: "<SPK1>"}

    for turn in turns:
        text = turn["value"].strip().replace("\n", " ")
        if text:
            dialog.append(f"{speaker_map[turn['from']]} {text}")

    if dialog:
        dialogs.append(" ".join(dialog))

###############################################


dataset = load_dataset("empathetic_dialogues", split="train")
conversations = defaultdict(list)
for sample in dataset:
    conversations[sample["conv_id"]].append((sample["utterance_idx"], sample["speaker_idx"], sample["utterance"]))

for conv in tqdm(conversations.values(), desc="Parsing EmpatheticDialogues"):
    sorted_conv = sorted(conv, key=lambda x: x[0])
    if len(sorted_conv) < 2:
        continue

    speaker_ids = list({s for _, s, _ in sorted_conv})
    if len(speaker_ids) != 2:
        continue

    if random.random() > 0.5:
        speaker_map = {speaker_ids[0]: "<SPK1>", speaker_ids[1]: "<SPK2>"}
    else:
        speaker_map = {speaker_ids[0]: "<SPK2>", speaker_ids[1]: "<SPK1>"}

    dialog = []
    for _, speaker_id, utterance in sorted_conv:
        text = utterance.strip().replace("\n", " ")
        if text:
            dialog.append(f"{speaker_map[speaker_id]} {text}")

    if dialog:
        dialogs.append(" ".join(dialog))


class DialogDataset(Dataset):
    def __init__(self, dialogs, tokenizer, max_length=512):
        self.inputs = []
        for dialog in tqdm(dialogs, desc="Tokenizing dialogs"):
            enc = tokenizer(dialog, truncation=True, max_length=max_length, return_tensors="pt")
            self.inputs.append(enc.input_ids.squeeze(0))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        ids = self.inputs[idx]
        return ids, ids  # input and label are same for LM

dataset = DialogDataset(dialogs, tokenizer)
loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=lambda x: tokenizer.pad(
    [{"input_ids": i[0]} for i in x],
    return_tensors="pt",
    padding=True
))


Tokenizing dialogs: 100%|██████████| 26408/26408 [00:15<00:00, 1754.36it/s]


In [6]:
print(dialogs[100])

<SPK2> hi how are you doing over there on that side of the internets <SPK1> *takes a step back* Hi! Hope you're doing well! So are you a fan of The Walt Disney Company? One of my favorite characters is Minnie Mouse and I was surprised to recently find out her name is short for Minerva Mouse! <SPK2> i like disney though i havent watched a disney film in a while. i do like minnie. she is cute as the dickens. <SPK1> Well there's actually a hidden subculture dedicated to finding hidden Mickey mouse images in all things Disney, you should definitely check out! <SPK2> i guess if you find a mickey, you win something? if i find one, i guess i should tell others and not keep it secret. <SPK1> That's a good point! Ha! Unless they change the locations up!  Are you also a fan of Bill Nye? He actually served as a science consultant on Flubber, believe it or not! <SPK2> i guess he knows a bit about the science of hypothetical substances. he is an entertaining fellow for sure. <SPK1> *raises eyebrows

In [7]:
model.train()
epochs = 2

for epoch in range(epochs):
    total_loss = 0
    loop = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        labels = input_ids.clone()

        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} complete — Average Loss: {avg_loss:.4f}")
    model.save_pretrained("fine-tuned-gpt2-speakers-8")
    tokenizer.save_pretrained("fine-tuned-gpt2-speakers-8")


Epoch 1:   0%|          | 0/3301 [00:00<?, ?it/s]You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
2025-05-11 10:03:40.308275: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746957820.323033     542 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746957820.327389     542 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746957820.339739     542 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target mor

Epoch 1 complete — Average Loss: 1.2058


Epoch 2: 100%|██████████| 3301/3301 [25:43<00:00,  2.14it/s, loss=1.17] 


Epoch 2 complete — Average Loss: 1.1657
